# Implementasi SVM untuk Klasifikasi Kelayakan PKH

**Deskripsi:** Notebook ini melakukan training model SVM untuk mengklasifikasikan
kelayakan calon penerima bantuan Program Keluarga Harapan (PKH) berdasarkan
8 atribut sosial-ekonomi. Data di-generate langsung di notebook ini.

**Output:** Model SVM (.pkl) siap integrasi ke web SPK


## 1. Import Library


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report, ConfusionMatrixDisplay,
                             roc_curve, auc, roc_auc_score)

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
print('Libraries loaded successfully!')


## 2. Generate Dataset Sintetis PKH

Dataset dibuat berdasarkan karakteristik dari penelitian lapangan di Kecamatan Kasimbar:
Desa Posona, Kasimbar Palapi, dan Posona Atas.


In [ ]:
np.random.seed(42)
N = 500

# Fitur
penghasilan = np.random.randint(200000, 8000000, N)
pekerjaan_opts = ['Tidak Bekerja', 'Buruh Tani', 'Nelayan', 'Petani', 'Buruh Bangunan',
                   'Pedagang Kecil', 'Supir', 'IRT', 'PNS', 'Karyawan Swasta', 'Pedagang Besar']
pekerjaan = np.random.choice(pekerjaan_opts, N, p=[0.15, 0.15, 0.10, 0.10, 0.10, 0.10, 0.05, 0.10, 0.02, 0.08, 0.05])
aset_opts = ['Tidak Punya', 'Rumah Sangat Sederhana', 'Rumah Sederhana',
             'Rumah + Lahan Terbatas', 'Rumah + Lahan Luas', 'Rumah + Kendaraan']
aset = np.random.choice(aset_opts, N, p=[0.15, 0.25, 0.25, 0.15, 0.10, 0.10])
ibu_hamil = np.random.choice([0, 1], N, p=[0.75, 0.25])
anak_usia_dini = np.clip(np.random.poisson(0.8, N), 0, 5)
anak_sekolah = np.clip(np.random.poisson(1.2, N), 0, 6)
disabilitas = np.random.choice([0, 1], N, p=[0.88, 0.12])
lansia = np.clip(np.random.poisson(0.3, N), 0, 3)
desa_opts = ['Posona', 'Kasimbar Palapi', 'Posona Atas']
desa = np.random.choice(desa_opts, N)

# Label (Layak/Tidak Layak) berdasarkan rules
layak = np.zeros(N, dtype=int)
for i in range(N):
    score = 0
    if penghasilan[i] < 800000: score += 3
    elif penghasilan[i] < 1500000: score += 2
    elif penghasilan[i] < 2500000: score += 1
    if pekerjaan[i] in ['Tidak Bekerja', 'Buruh Tani', 'Nelayan', 'IRT']: score += 2
    elif pekerjaan[i] in ['Buruh Bangunan', 'Pedagang Kecil', 'Supir']: score += 1
    if aset[i] in ['Tidak Punya', 'Rumah Sangat Sederhana']: score += 2
    elif aset[i] in ['Rumah Sederhana']: score += 1
    if ibu_hamil[i] == 1: score += 2
    score += min(anak_usia_dini[i], 2)
    score += min(anak_sekolah[i], 2)
    if disabilitas[i] == 1: score += 3
    score += min(lansia[i], 2)
    if score >= 5: layak[i] = 1

status = ['Layak' if l == 1 else 'Tidak Layak' for l in layak]

df = pd.DataFrame({
    'id': range(1, N+1),
    'nama': [f'KK-{i:03d}' for i in range(1, N+1)],
    'desa': desa,
    'penghasilan': penghasilan,
    'pekerjaan': pekerjaan,
    'kepemilikan_aset': aset,
    'ibu_hamil': ibu_hamil,
    'anak_usia_dini': anak_usia_dini,
    'anak_sekolah': anak_sekolah,
    'disabilitas': disabilitas,
    'lansia': lansia,
    'status_kelayakan': status
})

print(f'Dataset: {len(df)} data, {len(df.columns)} kolom')
print(f'Layak: {sum(layak)} | Tidak Layak: {N - sum(layak)}')
print(f'\n5 data pertama:')
display(df.head())


## 3. Preprocessing Data


In [ ]:
X = df.drop(['id', 'nama', 'desa', 'status_kelayakan'], axis=1)
y = df['status_kelayakan']

le_target = LabelEncoder()
y_encoded = le_target.fit_transform(y)
print(f'Target mapping: Layak=1, Tidak Layak=0')

categorical_features = ['pekerjaan', 'kepemilikan_aset', 'ibu_hamil', 'disabilitas']
numerical_features = ['penghasilan', 'anak_usia_dini', 'anak_sekolah', 'lansia']

le_dict = {}
for col in categorical_features:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    le_dict[col] = le
    print(f'{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)
print(f'\nTrain: {X_train.shape}, Test: {X_test.shape}')

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])


## 4. Training SVM dengan 3 Kernel


In [ ]:
results = []
kernels = ['linear', 'rbf', 'poly']
for kernel in kernels:
    svm = SVC(kernel=kernel, random_state=42, probability=True)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    results.append({
        'kernel': kernel, 'model': svm,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred)
    })
    print(f'{kernel.upper():10} | Akurasi: {results[-1]["accuracy"]:.4f} | Presisi: {results[-1]["precision"]:.4f} | Recall: {results[-1]["recall"]:.4f}')


## 5. GridSearchCV untuk Parameter Terbaik


In [ ]:
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.1, 0.01],
    'kernel': ['rbf', 'poly', 'sigmoid']
}
grid_search = GridSearchCV(SVC(random_state=42, probability=True), param_grid,
                           cv=5, scoring='accuracy', n_jobs=-1, verbose=0)
grid_search.fit(X_train_scaled, y_train)

best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_scaled)
y_prob_best = best_svm.predict_proba(X_test_scaled)[:, 1]

acc = accuracy_score(y_test, y_pred_best)
prec = precision_score(y_test, y_pred_best)
rec = recall_score(y_test, y_pred_best)
f1 = f1_score(y_test, y_pred_best)
auc_score = roc_auc_score(y_test, y_prob_best)

print(f'Parameter terbaik: {grid_search.best_params_}')
print(f'Akurasi: {acc:.2%} | Presisi: {prec:.2%} | Recall: {rec:.2%} | F1: {f1:.2%} | AUC: {auc_score:.4f}')
print(classification_report(y_test, y_pred_best, target_names=['Tidak Layak', 'Layak']))


## 6. Confusion Matrix


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_best,
    display_labels=['Tidak Layak', 'Layak'], cmap='Blues', values_format='d', ax=ax)
ax.set_title(f'Confusion Matrix - SVM {best_svm.kernel}\nAkurasi: {acc:.2%}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150)
plt.show()


## 7. Simpan Model (.pkl)


In [ ]:
class SVMPipeline:
    def __init__(self, scaler, label_encoders, target_encoder, svm_model, numerical_features):
        self.scaler = scaler
        self.label_encoders = label_encoders
        self.target_encoder = target_encoder
        self.svm_model = svm_model
        self.numerical_features = numerical_features

    def preprocess(self, X_input):
        X = X_input.copy()
        for col, le in self.label_encoders.items():
            if col in X.columns:
                X[col] = X[col].apply(lambda x: x if x in le.classes_ else le.classes_[0])
                X[col] = le.transform(X[col])
        X[self.numerical_features] = self.scaler.transform(X[self.numerical_features])
        return X

    def predict(self, X_input):
        return self.svm_model.predict(self.preprocess(X_input))

    def predict_proba(self, X_input):
        return self.svm_model.predict_proba(self.preprocess(X_input))

    def predict_label(self, X_input):
        return self.target_encoder.inverse_transform(self.predict(X_input))

pipeline = SVMPipeline(scaler, le_dict, le_target, best_svm, numerical_features)

model_path = '/kaggle/working/model_svm_pkh.pkl'
joblib.dump(pipeline, model_path)
print(f'Model saved: {model_path}')
print(f'Size: {os.path.getsize(model_path)/1024:.2f} KB')


## 8. Demo Prediksi


In [ ]:
demo = pd.DataFrame([
    {'penghasilan': 500000, 'pekerjaan': 'Tidak Bekerja', 'kepemilikan_aset': 'Tidak Punya', 'ibu_hamil': 1, 'anak_usia_dini': 2, 'anak_sekolah': 1, 'disabilitas': 0, 'lansia': 0},
    {'penghasilan': 5000000, 'pekerjaan': 'PNS', 'kepemilikan_aset': 'Rumah + Kendaraan', 'ibu_hamil': 0, 'anak_usia_dini': 0, 'anak_sekolah': 0, 'disabilitas': 0, 'lansia': 0},
    {'penghasilan': 1200000, 'pekerjaan': 'Buruh Tani', 'kepemilikan_aset': 'Rumah Sangat Sederhana', 'ibu_hamil': 0, 'anak_usia_dini': 1, 'anak_sekolah': 2, 'disabilitas': 1, 'lansia': 1},
])
labels, probas = pipeline.predict_label(demo), pipeline.predict_proba(demo)
for i, (label, proba) in enumerate(zip(labels, probas)):
    print(f'Data {i+1}: {label} (Layak: {proba[1]:.1%}, Tidak: {proba[0]:.1%})')


## 9. Ringkasan
Model SVM siap diintegrasikan ke web SPK Flask. Pipeline preprocessing sudah termasuk dalam .pkl.
